# Entrenamiento y evaluación en Colab

Notebook principal nuevo del proyecto.

In [ ]:
# ============================================================
# SETUP — run this cell once after every runtime restart
# Do NOT run training here. Select one experiment cell below.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

# --- Environment fingerprint (for reproducibility debugging) ---
import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!pip show albumentations | grep Version
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"
# --------------------------------------------------------------

import sys
sys.path.append("/content/tesis-seg")

from src.defaults import get_default_config, summarize_config
from src.augmentations import (
    get_supervised_train_augmentation,
    get_weak_augmentation,
    get_strong_augmentation,
)
from src.datasets import (
    build_supervised_datasets,
    build_unlabeled_datasets,
    build_dataloaders,
)
from src.train import run_training
from src.evaluate import evaluate_checkpoint
from src.visualization import show_dataset_examples, show_predictions

# ── Debug fingerprint helpers ─────────────────────────────────
import json, os, platform, subprocess, time, importlib.metadata

def _fp_get_version(pkg):
    try: return importlib.metadata.version(pkg)
    except Exception: return None

def _fp_git_hash_from_src(src_train_file):
    # Derive repo root from src/train.py: {repo_root}/src/train.py
    repo_root = os.path.dirname(os.path.dirname(os.path.abspath(src_train_file)))
    try:
        return subprocess.check_output(
            ["git", "log", "--oneline", "-1"], cwd=repo_root, stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception: return None

def _fp_save(pre, post, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump({"pre_run": pre, "post_run": post}, f, indent=2, default=str)

def _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                    unlabeled_ds=None, temporal_unlab_ds=None):
    import torch, sys
    from src.models import create_model

    # 1. Environment
    try: gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
    except: gpu_name = "n/a"
    env = {
        "python":      sys.version,
        "torch":       torch.__version__,
        "torchvision": _fp_get_version("torchvision"),
        "cuda":        torch.version.cuda,
        "cudnn":       str(torch.backends.cudnn.version()) if torch.cuda.is_available() else "n/a",
        "segmentation_models_pytorch": _fp_get_version("segmentation-models-pytorch"),
        "albumentations": _fp_get_version("albumentations"),
        "platform":    platform.platform(),
        "gpu_name":    gpu_name,
    }

    # 2. Code provenance — git hash derived from actual runtime src path
    import src.train, src.datasets, src.defaults, src.evaluate
    provenance = {
        "src_train":    src.train.__file__,
        "src_datasets": src.datasets.__file__,
        "src_defaults": src.defaults.__file__,
        "src_evaluate": src.evaluate.__file__,
        "git_hash":     _fp_git_hash_from_src(src.train.__file__),
    }

    # 3. Effective config
    cfg_keys = [
        "seed", "arch", "backbone", "n_classes",
        "image_preproc", "mask_smoothing", "target_size", "use_pad", "imagenet_norm",
        "batch_size", "num_workers", "drop_last", "num_augmented",
        "lr", "weight_decay", "epochs", "warmup_epochs", "patience_es", "eval_threshold",
        "use_semi", "use_temp_consistency",
        "lambda_u", "tau", "ema_decay", "semi_start_epoch", "semi_warmup_epochs", "lambda_t",
        "unlabeled_subdir", "exp_dir",
    ]
    eff_cfg = {k: cfg.get(k) for k in cfg_keys}
    unlab_loader = loaders.get("unlabeled_loader")
    eff_cfg["batch_size_unlab"] = unlab_loader.batch_size if unlab_loader is not None else None

    # 4. Dataset / loader facts
    ds_facts = {
        "len_train_ds":          len(train_ds),
        "len_val_ds":            len(val_ds),
        "len_test_ds":           len(test_ds),
        "len_unlabeled_ds":      len(unlabeled_ds) if unlabeled_ds is not None else None,
        "len_temporal_unlab_ds": len(temporal_unlab_ds) if temporal_unlab_ds is not None else None,
        "train_loader_batch_size":      loaders["train_loader"].batch_size,
        "train_loader_num_workers":     loaders["train_loader"].num_workers,
        "train_loader_drop_last":       loaders["train_loader"].drop_last,
        "unlabeled_loader_batch_size":  unlab_loader.batch_size if unlab_loader else None,
        "unlabeled_loader_num_workers": unlab_loader.num_workers if unlab_loader else None,
        "unlabeled_loader_drop_last":   unlab_loader.drop_last if unlab_loader else None,
    }

    # 5. Sample identifiers — reads .files attribute, no IO beyond what dataset already did
    try: sup_ids = train_ds.files[:5]
    except Exception as e: sup_ids = f"unavailable: {e}"
    try: unl_ids = unlabeled_ds.files[:5] if unlabeled_ds is not None else None
    except Exception as e: unl_ids = f"unavailable: {e}"
    sample_ids = {"first5_train": sup_ids, "first5_unlabeled": unl_ids}

    # 6. Batch tensor shapes — analytical, no DataLoader consumed, no RNG touched
    try:
        H, W = cfg["target_size"]
        C = 3  # IMREAD_COLOR: grayscale PNGs expand to 3 identical channels
        eff_bs = cfg["batch_size"] * (1 + cfg.get("num_augmented", 0))  # flatten_collate
        bs_u = max(1, cfg["batch_size"] // 4)  # mirrors datasets.py build_dataloaders
        batch_shapes = {
            "xb":   [eff_bs, C, H, W],
            "yb":   [eff_bs, 1, H, W],
            "xw_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "xs_u": [bs_u, C, H, W] if unlab_loader is not None else None,
            "note": "analytically derived from cfg — no DataLoader consumed",
        }
    except Exception as e:
        batch_shapes = {"error": str(e)}

    # 7. Model fingerprint — RNG save/restore so training is unaffected.
    # Belt-and-suspenders: run_training() also calls seed_everything(seed) first.
    try:
        import random as _random, numpy as _np
        _rng = {
            "py":   _random.getstate(),
            "np":   _np.random.get_state(),
            "th":   torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        model_fp = {
            "total_params":          sum(p.numel() for p in _m.parameters()),
            "trainable_params":      sum(p.numel() for p in _m.parameters() if p.requires_grad),
            "first_state_dict_keys": list(_m.state_dict().keys())[:8],
        }
        del _m
        _random.setstate(_rng["py"])
        _np.random.set_state(_rng["np"])
        torch.set_rng_state(_rng["th"])
        if _rng["cuda"] is not None:
            torch.cuda.set_rng_state_all(_rng["cuda"])
    except Exception as e:
        model_fp = {"error": str(e)}

    return {
        "timestamp_utc":     time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "environment":       env,
        "provenance":        provenance,
        "effective_cfg":     eff_cfg,
        "dataset_facts":     ds_facts,
        "sample_ids":        sample_ids,
        "batch_shapes":      batch_shapes,
        "model_fingerprint": model_fp,
    }


def _fp_collect_post(artifacts, results):
    history = artifacts.get("history") or []
    best_row = max(history, key=lambda r: r.get("val_iou_global", 0.0)) if history else None
    vm = (results or {}).get("val_metrics", {})
    tm = (results or {}).get("test_metrics", {})
    return {
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "best_path":     artifacts.get("best_path"),
        "best_epoch_info": {
            "epoch":          best_row.get("epoch") if best_row else None,
            "val_iou_global": best_row.get("val_iou_global") if best_row else None,
            "val_loss":       best_row.get("val_loss") if best_row else None,
        },
        "val_metrics": {
            "f1_global":       vm.get("global_f1"),
            "iou_global":      vm.get("global_iou"),
            "f1_sample_mean":  vm.get("sample_mean_f1"),
            "iou_sample_mean": vm.get("sample_mean_iou"),
        },
        "test_metrics": {
            "f1_global":       tm.get("global_f1"),
            "iou_global":      tm.get("global_iou"),
            "f1_sample_mean":  tm.get("sample_mean_f1"),
            "iou_sample_mean": tm.get("sample_mean_iou"),
        },
        "finished_successfully": True,
        "exception": None,
    }
# ──────────────────────────────────────────────────────────────

print("Imports OK — select one experiment cell below and run it.")

## Experiment: supervised

Fully supervised baseline — no unlabeled data used.  
Temporal branch: **OFF** (`use_semi=False`, `use_temp_consistency=False`).

In [ ]:
# =========================
# supervised
# =========================
cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/supervised"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Supervised only — no unlabeled data
cfg["seed"]                 = 0
cfg["use_semi"]             = False
cfg["use_temp_consistency"] = False
cfg["lambda_t"]             = 0.0

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5
cfg["run_ruler_eval"] = True

print(summarize_config(cfg))

train_tf = get_supervised_train_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=None,
    temporal_unlab_ds=None,
)

# ── Debug fingerprint: pre_run ────────────────────────────────
_fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
_fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                            unlabeled_ds=None, temporal_unlab_ds=None)
_fp_save(_fp_pre, None, _fp_path)
print(f"[fingerprint] pre_run saved → {_fp_path}")
# ─────────────────────────────────────────────────────────────

try:
    artifacts = run_training(cfg, loaders)

    results = evaluate_checkpoint(
        cfg,
        artifacts['model'],
        loaders,
        artifacts['best_path'],
        artifacts['history'],
    )

    # ── Debug fingerprint: post_run ───────────────────────────
    _fp_post = _fp_collect_post(artifacts, results)
    _fp_save(_fp_pre, _fp_post, _fp_path)
    print(f"[fingerprint] post_run saved → {_fp_path}")
    # ─────────────────────────────────────────────────────────
    print(results)

except Exception as _fp_exc:
    import traceback
    _fp_save(_fp_pre, {
        "finished_successfully": False,
        "exception": traceback.format_exc(),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }, _fp_path)
    print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
    raise

## Experiment: semi_std_matched_r3

Random matched control for `semi_r3`.  
Pool: `unlabeling_std_matched_r3/images` — same per-video count as r=3, uniform random selection, no temporal constraint.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_std_matched_r3
# =========================
cfg = get_default_config()

# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_std_matched_r3"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: random control matched to r=3 per-video counts
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r3/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

# ── Debug fingerprint: pre_run ────────────────────────────────
_fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
_fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                            unlabeled_ds=unlabeled_ds,
                            temporal_unlab_ds=temporal_unlab_ds)
_fp_save(_fp_pre, None, _fp_path)
print(f"[fingerprint] pre_run saved → {_fp_path}")
# ─────────────────────────────────────────────────────────────

try:
    artifacts = run_training(cfg, loaders)

    results = evaluate_checkpoint(
        cfg,
        artifacts['model'],
        loaders,
        artifacts['best_path'],
        artifacts['history'],
    )

    # ── Debug fingerprint: post_run ───────────────────────────
    _fp_post = _fp_collect_post(artifacts, results)
    _fp_save(_fp_pre, _fp_post, _fp_path)
    print(f"[fingerprint] post_run saved → {_fp_path}")
    # ─────────────────────────────────────────────────────────
    print(results)

except Exception as _fp_exc:
    import traceback
    _fp_save(_fp_pre, {
        "finished_successfully": False,
        "exception": traceback.format_exc(),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }, _fp_path)
    print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
    raise

## Experiment: semi_std_matched_r10

Random matched control for `semi_r10`.  
Pool: `unlabeling_std_matched_r10/images` — same per-video count as r=10, uniform random selection, no temporal constraint.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_std_matched_r10
# =========================
cfg = get_default_config()

# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_std_matched_r10"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: random control matched to r=10 per-video counts
cfg["unlabeled_subdir"] = "unlabeling_std_matched_r10/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

# ── Debug fingerprint: pre_run ────────────────────────────────
_fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
_fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                            unlabeled_ds=unlabeled_ds,
                            temporal_unlab_ds=temporal_unlab_ds)
_fp_save(_fp_pre, None, _fp_path)
print(f"[fingerprint] pre_run saved → {_fp_path}")
# ─────────────────────────────────────────────────────────────

try:
    artifacts = run_training(cfg, loaders)

    results = evaluate_checkpoint(
        cfg,
        artifacts['model'],
        loaders,
        artifacts['best_path'],
        artifacts['history'],
    )

    # ── Debug fingerprint: post_run ───────────────────────────
    _fp_post = _fp_collect_post(artifacts, results)
    _fp_save(_fp_pre, _fp_post, _fp_path)
    print(f"[fingerprint] post_run saved → {_fp_path}")
    # ─────────────────────────────────────────────────────────
    print(results)

except Exception as _fp_exc:
    import traceback
    _fp_save(_fp_pre, {
        "finished_successfully": False,
        "exception": traceback.format_exc(),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }, _fp_path)
    print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
    raise

## Experiment: semi_r3

Temporal-neighbor semi-supervised run with r=3.  
Pool: `unlabeling_r3_max0/images`.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_r3
# =========================
cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_r3"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: temporal-neighbor r=3
cfg["unlabeled_subdir"] = "unlabeling_r3_max0/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

# ── Debug fingerprint: pre_run ────────────────────────────────
_fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
_fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                            unlabeled_ds=unlabeled_ds,
                            temporal_unlab_ds=temporal_unlab_ds)
_fp_save(_fp_pre, None, _fp_path)
print(f"[fingerprint] pre_run saved → {_fp_path}")
# ─────────────────────────────────────────────────────────────

try:
    artifacts = run_training(cfg, loaders)

    results = evaluate_checkpoint(
        cfg,
        artifacts['model'],
        loaders,
        artifacts['best_path'],
        artifacts['history'],
    )

    # ── Debug fingerprint: post_run ───────────────────────────
    _fp_post = _fp_collect_post(artifacts, results)
    _fp_save(_fp_pre, _fp_post, _fp_path)
    print(f"[fingerprint] post_run saved → {_fp_path}")
    # ─────────────────────────────────────────────────────────
    print(results)

except Exception as _fp_exc:
    import traceback
    _fp_save(_fp_pre, {
        "finished_successfully": False,
        "exception": traceback.format_exc(),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }, _fp_path)
    print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
    raise

## Experiment: semi_r10

Temporal-neighbor semi-supervised run with r=10.  
Pool: `unlabeling_r10_max0/images`.  
Temporal branch: **OFF** (`use_temp_consistency=False`, `lambda_t=0.0`).

In [ ]:
# =========================
# semi_r10
# =========================
cfg = get_default_config()
# Paths
cfg["img_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["msk_root"]    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
cfg["rotulos_dir"] = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
cfg["exp_dir"] = "/content/drive/MyDrive/UNM_vertebras_seg_v3/runs/semi_r10"

# Architecture
cfg["arch"]      = "unetpp"
cfg["backbone"]  = "efficientnet-b3"
cfg["n_classes"] = 1

# Semi-supervised (standard branch ON; temporal branch OFF)
cfg["seed"]                 = 0
cfg["use_semi"]             = True
cfg["use_temp_consistency"] = False   # temporal branch OFF
cfg["lambda_u"]             = 0.05
cfg["tau"]                  = 0.95
cfg["ema_decay"]            = 0.99
cfg["semi_start_epoch"]     = 30
cfg["semi_warmup_epochs"]   = 20
cfg["lambda_t"]             = 0.0    # explicitly zero

# Preprocessing
cfg["image_preproc"]  = "base"
cfg["mask_smoothing"] = "none"
cfg["use_fixed_crop"] = False
cfg["target_size"]    = (320, 320)
cfg["use_pad"]        = True
cfg["imagenet_norm"]  = False

# Training
cfg["batch_size"]     = 5
cfg["num_workers"]    = 4
cfg["drop_last"]      = True
cfg["num_augmented"]  = 5
cfg["lr"]             = 1e-3
cfg["weight_decay"]   = 1e-4
cfg["epochs"]         = 2000
cfg["warmup_epochs"]  = 10
cfg["patience_es"]    = 20
cfg["eval_threshold"] = 0.5

# Unlabeled pool: temporal-neighbor r=10
cfg["unlabeled_subdir"] = "unlabeling_r10_max0/images"
cfg["run_ruler_eval"]   = True

print(summarize_config(cfg))

train_tf  = get_supervised_train_augmentation(cfg)
weak_tf   = get_weak_augmentation(cfg)
strong_tf = get_strong_augmentation(cfg)

train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(
    cfg, weak_tf=weak_tf, strong_tf=strong_tf
)

loaders = build_dataloaders(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,
    unlabeled_ds=unlabeled_ds,
    temporal_unlab_ds=temporal_unlab_ds,
)

# ── Debug fingerprint: pre_run ────────────────────────────────
_fp_path = os.path.join(cfg["exp_dir"], "debug_fingerprint.json")
_fp_pre  = _fp_collect_pre(cfg, loaders, train_ds, val_ds, test_ds,
                            unlabeled_ds=unlabeled_ds,
                            temporal_unlab_ds=temporal_unlab_ds)
_fp_save(_fp_pre, None, _fp_path)
print(f"[fingerprint] pre_run saved → {_fp_path}")
# ─────────────────────────────────────────────────────────────

try:
    artifacts = run_training(cfg, loaders)

    results = evaluate_checkpoint(
        cfg,
        artifacts['model'],
        loaders,
        artifacts['best_path'],
        artifacts['history'],
    )

    # ── Debug fingerprint: post_run ───────────────────────────
    _fp_post = _fp_collect_post(artifacts, results)
    _fp_save(_fp_pre, _fp_post, _fp_path)
    print(f"[fingerprint] post_run saved → {_fp_path}")
    # ─────────────────────────────────────────────────────────
    print(results)

except Exception as _fp_exc:
    import traceback
    _fp_save(_fp_pre, {
        "finished_successfully": False,
        "exception": traceback.format_exc(),
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }, _fp_path)
    print(f"[fingerprint] run FAILED — partial fingerprint saved → {_fp_path}")
    raise

In [ ]:
# === Posthoc: C2-C4 ruler comparison from existing exp_dir ===
# Run this cell independently to generate c2c4_comparison.csv and c2c4_vis/
# from any completed experiment folder, without retraining.
# Safe to run on seed-specific folders (runs/semi_r3_seed0, etc.).

import json, os
from src.ruler_eval import compare_c2c4_manual_vs_auto, visualize_c2c4_comparison

# ── Set this to the target run folder ──────────────────────────────────────
_exp_dir = "/content/drive/MyDrive/UNM_vertebras_seg_v3/experiments/EXPERIMENT_NAME"
# ───────────────────────────────────────────────────────────────────────────

with open(os.path.join(_exp_dir, "config.json"), "r") as _f:
    _cfg = json.load(_f)

_rotulos_dir    = _cfg["rotulos_dir"]
_img_root       = _cfg["img_root"]
_target_size    = tuple(_cfg.get("target_size", [320, 320]))
_pred_masks_dir = os.path.join(_exp_dir, "test_preds")
_test_images_dir = os.path.join(_img_root, "test", "images")

compare_c2c4_manual_vs_auto(
    rotulos_dir=_rotulos_dir,
    pred_masks_dir=_pred_masks_dir,
    out_csv=os.path.join(_exp_dir, "c2c4_comparison.csv"),
    target_size=_target_size,
)

visualize_c2c4_comparison(
    rotulos_dir=_rotulos_dir,
    pred_masks_dir=_pred_masks_dir,
    test_images_dir=_test_images_dir,
    out_dir=os.path.join(_exp_dir, "c2c4_vis"),
    target_size=_target_size,
)
